In [21]:
import warnings
warnings.filterwarnings("ignore")
import os

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langchain_community.tools import DuckDuckGoSearchResults
from langgraph.prebuilt import create_react_agent

claude = ChatAnthropic(model="claude-sonnet-4-5", temperature=0)
search = DuckDuckGoSearchResults(num_results=5)

print("Ready!")

Ready!


In [22]:
import httpx
from bs4 import BeautifulSoup
from langchain_core.tools import tool

#Tool for webpage scraping
@tool
def scrape_webpage(url: str) -> str:
    """Scrape the full text content of a webpage given its URL."""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
        response = httpx.get(url, headers=headers, timeout=10, follow_redirects=True)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Remove noise
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        
        # Extract clean text
        text = soup.get_text(separator="\n", strip=True)
        
        # Limit to first 3000 chars to avoid overwhelming the LLM
        return text[:3000] if len(text) > 3000 else text
    
    except Exception as e:
        return f"Error scraping {url}: {str(e)}"

# Test scraping
result = scrape_webpage.invoke("https://techcrunch.com/category/artificial-intelligence/")
print(result[:500])

AI News & Artificial Intelligence | TechCrunch
–:–:–:–
Save up to $680 on your pass with Super Early Bird rates.
REGISTER NOW
.
Save up to $680 on your Disrupt 2026 pass. Ends February 27.
REGISTER NOW
.
Close
AI
News coverage on artificial intelligence and machine learning tech, the companies building them, and the ethical issues AI raises today. This encompasses
generative AI
, including large language models, text-to-image and text-to-video models; speech recognition and generation; and predi


In [23]:
import datetime

# Get the project root dynamically
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPORTS_DIR = os.path.join(PROJECT_ROOT, "deep-research-agent", "reports")

@tool
def save_report(content: str, filename: str = "") -> str:
    """Save a research report as a markdown file in the reports folder."""
    if not filename:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"report_{timestamp}.md"

    # Ensure reports directory exists
    os.makedirs(REPORTS_DIR, exist_ok=True)
    
    filepath = os.path.join(REPORTS_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    
    return f"Report saved to {filepath}"

# Verify the path is correct before testing
print("Reports will be saved to:", REPORTS_DIR)
# Test report
save_report.invoke({"content": "# Test Report\nThis is a test.", "filename": "test.md"})

Reports will be saved to: C:\Users\Henry\Documents\deep-research-agent\reports


'Report saved to C:\\Users\\Henry\\Documents\\deep-research-agent\\reports\\test.md'

In [24]:
tools = [search, scrape_webpage, save_report]

agent = create_react_agent(claude, tools=tools)

print("Agent ready with tools:", [t.name for t in tools])

Agent ready with tools: ['duckduckgo_results_json', 'scrape_webpage', 'save_report']


In [25]:
query = """
You are a research agent. Follow these steps:
1. Search for 'LangGraph vs CrewAI vs AutoGen comparison 2026'
2. Pick the 2 most relevant URLs from the results and scrape them for details
3. Synthesize a structured report with these sections:
   - Overview of each framework
   - Key differences
   - Best use cases for each
   - Recommendation for a beginner building a research agent
4. Save the report as 'agent_frameworks_comparison.md'
"""

result = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

print(result["messages"][-1].content)

Perfect! I've successfully completed your research request. Here's what I did:

## Summary of Research Process

1. **Searched** for "LangGraph vs CrewAI vs AutoGen comparison 2026" and found several relevant sources

2. **Scraped the 2 most relevant URLs:**
   - Latenode's comprehensive technical comparison
   - OpenAgents.org's detailed framework analysis
   - Also gathered information from DataMites and LinkedIn articles

3. **Synthesized a comprehensive report** with the following sections:
   - **Overview of each framework** - Detailed descriptions of LangGraph, CrewAI, and AutoGen including their architecture, features, and technical capabilities
   - **Key differences** - Comparison tables covering architecture philosophy, control vs. simplicity, state management, learning curve, and protocol support
   - **Best use cases for each** - Specific scenarios where each framework excels with real-world examples
   - **Recommendation for beginners** - Detailed guidance recommending Crew

In [26]:
with open("../reports/agent_frameworks_comparison.md", "r", encoding="utf-8") as f:
    print(f.read())

# AI Agent Frameworks Comparison 2026: LangGraph vs CrewAI vs AutoGen

## Executive Summary

This report provides a comprehensive comparison of three leading multi-agent AI frameworks in 2026: LangGraph, CrewAI, and AutoGen. Each framework offers distinct approaches to building AI agent systems, with different philosophies, strengths, and ideal use cases. This analysis is designed to help developers, particularly beginners, make informed decisions when selecting a framework for building research agents and other AI applications.

---

## Overview of Each Framework

### LangGraph

**Philosophy:** Graph-based workflow orchestration with precise control

LangGraph is an open-source library within the LangChain ecosystem, specifically designed for building stateful, multi-actor applications powered by Large Language Models (LLMs). It treats AI workflows as directed graphs with nodes and edges, providing developers with granular control over task execution and state management.

**Core Char